# Lab 4: ReAct Agent


**Agentic AI** คือ ระบบ AI ที่ไม่ได้แค่ตอบคำถามแบบ Passive แต่สามารถตัดสินใจและลงมือทำ (Active) เพื่อให้บรรลุเป้าหมายที่ได้รับมอบหมาย 

**ReAct (Reason + Act)** คือ รูปแบบการคิดของ Agent แบบหนึ่ง ประกอบด้วย 2 ส่วนหลัก:
* Reason (Reasoning): การคิดวิเคราะห์ วางแผน และสรุปสถานการณ์ปัจจุบัน
* Act (Acting): การลงมือทำโดยใช้ Tools ที่มี หรือตอบกลับผู้ใช้

![https://mlpills.substack.com/p/diy-14-step-by-step-implementation](https://substackcdn.com/image/fetch/$s_!du7b!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F06f2cf46-df40-48f9-a798-931222b0f70a_590x592.png "https://mlpills.substack.com/p/diy-14-step-by-step-implementation")

ref: https://mlpills.substack.com/p/diy-14-step-by-step-implementation


ใน Lab นี้เราจะสร้าง ReACT Agent ที่สามารถ **วางแผน (Plan)**, **เลือกใช้เครื่องมือ (Action / Tool calling)**, อ่านผลลัพธ์จากเครื่องมือ (**Observation**) และ **ทบทวนความคืบหน้า (Reflection)** ก่อนตัดสินใจทำขั้นตอนถัดไป

อ่านเพิ่มเติม https://huggingface.co/blog/VirtualOasis/agents-vs-workflows-en


## เป้าหมาย

1. เข้าใจวงจร **ReAct: Reason → Act → Observe**
2. ให้ LLM ออกแบบ **Plan** ก่อนเริ่มทำงาน
3. ใช้ **Tool calling** เพื่อเรียกใช้เครื่องมือภายนอก
4. เพิ่ม Reflection เพื่อตรวจสอบหลักฐานและสิ่งที่ยังขาด
5. ป้องกัน agent loop ด้วย `max_steps` และจัดการ tool error

```mermaid
flowchart LR
    A[User Question] --> B[Plan]
    B --> C[Agent / Reason]
    C -->|tool_calls| D[Act: Run Tools]
    D --> E[Observe]
    E --> F[Reflect]
    F --> C
    C -->|no tool call| G[Final Answer]
```

In [1]:
!uv pip install -q python-dotenv pandas langgraph langchain-google-genai

In [2]:
import json
import os
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
MODEL_NAME = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(model=MODEL_NAME, api_key=GEMINI_KEY)

## Step 1: สร้าง Tools

ตัวอย่างนี้ใช้ข้อมูลสินค้าจาก Shoppe และแบ่งความสามารถออกเป็น 3 tools:

| Tool | หน้าที่ | เหตุผลที่แยกออกมา |
| --- | --- | --- |
| `search_products` | ค้นหา `product_id` จากคำสำคัญ | Search tool ส่งข้อมูลเท่าที่จำเป็นกลับมา |
| `get_product_details` | อ่านราคา คะแนน และ stock จาก ID | Agent ต้องหา ID ก่อน แล้วจึงอ่านรายละเอียด |
| `calculator` | คำนวณเลขด้วย operation ที่กำหนด | ไม่ให้ LLM เดาผลคำนวณเอง |

In [6]:
products = pd.read_csv("Examples/shopee-products.csv")

def to_json(data):
    return json.dumps(data, ensure_ascii=False, default=str)

@tool
def search_products(keyword: str, limit: int = 5) -> str:
    """ค้นหาสินค้าจากคำที่อยู่ในชื่อสินค้า ใช้ก่อนเมื่อต้องหา product ID และห้ามเดา ID เอง"""
    limit = max(1, min(int(limit), 10))
    matched = products[
        products["title"].str.contains(keyword, case=False, na=False, regex=False)
    ].head(limit)
    results = [
        {"product_id": str(row["id"]), "title": row["title"]}
        for _, row in matched.iterrows()
    ]
    return to_json({"keyword": keyword, "count": len(results), "products": results})

@tool
def get_product_details(product_id: str) -> str:
    """อ่านราคา คะแนน stock และผู้ขายของสินค้าหนึ่งชิ้นจาก product ID ที่ได้จาก search_products"""
    matched = products[products["id"].astype(str) == str(product_id)]
    if matched.empty:
        return to_json({"error": f"ไม่พบ product_id={product_id}"})
    row = matched.iloc[0]
    fields = ["id", "title", "final_price", "currency", "rating", "stock", "seller_name"]
    detail = {field: row[field] for field in fields if field in row.index and pd.notna(row[field])}
    detail["id"] = str(detail["id"])
    return to_json(detail)

@tool
def calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]) -> str:
    """คำนวณเลขสองจำนวน ใช้เมื่อต้องการผลรวม ผลต่าง ผลคูณ หรือผลหารที่แม่นยำ"""
    operations = {
        "add": lambda: a + b,
        "subtract": lambda: a - b,
        "multiply": lambda: a * b,
        "divide": lambda: a / b if b != 0 else None,
    }
    if operation not in operations:
        return to_json({"error": f"ไม่รองรับ operation={operation}"})
        
    result = operations[operation]()
    if result is None:
        return to_json({"error": "หารด้วยศูนย์ไม่ได้"})
    return to_json({"a": a, "b": b, "operation": operation, "result": result})

In [23]:
search_products.func("เสื้อ", 5)

'{"keyword": "เสื้อ", "count": 5, "products": [{"product_id": "24832368987", "title": "🌈SpinnyHouse🌈เสื้อครอปแต่งย่นข้าง ปักโบว์ช่วงอก BABY TEE 😻 ผ้ายูนิโคล่ยืดตามตัว ทรงน่ารัดใส่ได้ทุกวัน รุ่น ตาแป๋ว"}, {"product_id": "18995315342", "title": "Beatrice - Cream🌷✨🍦เสื้อสายเดี่ยวกับกระโปรง ��ไตล์ลูกคุณหนูคะ น่ารักมากๆค่ะ🫶🏻 BT-CREAM"}, {"product_id": "23042783319", "title": "BB002 เสื้อยืดเด็ก คุณหนู ลายดีสนีย์ 👧🏻👦🏻 คอกลม แขนสั้น"}, {"product_id": "20639839835", "title": "[S-5XL] เสื้อยืดผ้าฝ้าย พิมพ์ลายการ์ตูน Captain Haddock Tintin แฟชั่นฤดูร้อน สไตล์ฮิปฮอป สําหรับผู้ชาย"}, {"product_id": "20037438526", "title": "Yuedpao[ใหม่ล่าสุด]รุ่นโคตรนุ่ม นุ่มตั้งแต่กำเนิด ยืดแต่ไม่ย้วย ยับยาก เสื้อยืดคอกลม Set Cozy Nature"}]}'

In [24]:
get_product_details.func("20639839835")

'{"id": "20639839835", "title": "[S-5XL] เสื้อยืดผ้าฝ้าย พิมพ์ลายการ์ตูน Captain Haddock Tintin แฟชั่นฤดูร้อน สไตล์ฮิปฮอป สําหรับผู้ชาย", "final_price": 169.0, "currency": "THB", "rating": 0.0, "seller_name": "w9s███eaz███"}'

In [26]:
calculator.func(20, 45, "multiply")

'{"a": 20, "b": 45, "operation": "multiply", "result": 900}'

In [32]:
calculator.func(150, 54, "subtract")

'{"a": 150, "b": 54, "operation": "subtract", "result": 96}'

## Step 2: Bind Tools และสร้าง ToolNode

`@tool` แปลง Python function พร้อม type hints และ docstring ให้เป็น tool schema ที่ LLM เข้าใจ จากนั้น:

- `llm.bind_tools(tools)` ทำให้ LLM เลือกชื่อ tool และสร้าง arguments ได้
- `ToolNode(tools)` รับ `tool_calls`, เรียก Python functions และส่งผลกลับเป็น `ToolMessage` โดยอัตโนมัติ


ในกรณีที่ไม่ใช้ `@tool` จะต้องเขียน tool schema, dispatch dictionary หรือ loop สำหรับ execute tool calls เอง

In [28]:
tools = [search_products, get_product_details, calculator]

tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

In [34]:
# ทดลองเรียก tools โดยตรงก่อนให้ Agent เป็นผู้เลือกใช้
print(search_products.invoke({"keyword": "รองเท้า", "limit": 3}))

print(calculator.invoke({"a": 250, "b": 3, "operation": "multiply"}))

{"keyword": "รองเท้า", "count": 3, "products": [{"product_id": "20659907642", "title": "□❡รองเท้าแตะผู้หญิง Wnc Native Waterproof Shoes EVA Non-slip Beach Hollow Sandals Solid Color Couple"}, {"product_id": "19950623906", "title": "[ลูกค้าใหม่ราคา 1 บาท]🍎รองเท้านักเรียนโกลซิตี้ GCรุ่นFC001/Matin T205 ทนชาย หญิง น้ำตาล ขาว ดำ ไซร์31-45(มีบิลเบิกรรให้)"}, {"product_id": "25825858326", "title": "รองเท้าผ้าใบ KE-EN Original Running Shoes ZIONIC สีเขียว พร้อมส่ง"}]}
{"a": 250.0, "b": 3.0, "operation": "multiply", "result": 750.0}


## Step 3: Plan และ Reflection

การทำงานของ Agent เริ่มจาก
1. บอกให้ agent วางแผน ตามโจทย์ที่ให้ไป
2. หลังจากนั้น จะเข้าสู่ ReAct loop โดย agent จะเรียกใช้ tools เพื่อดำเนินการตามแผนที่วางไว้
3. agent จะเรียก reflection เพื่อตรวจว่ามีหลักฐานอะไรแล้ว ผลจาก tool มี error หรือไม่ และยังขาดข้อมูลอะไร

```mermaid
flowchart LR
    A[User Question] --> B[Plan]
    B --> C[Agent / Reason]
    C -->|tool_calls| D[Act: Run Tools]
    D --> E[Observe]
    E --> F[Reflect]
    F --> C
    C -->|no tool call| G[Final Answer]
    style B fill:#ff9999,stroke:#333
    style F fill:#ff9999,stroke:#333
```

In [37]:
class AgentState(MessagesState):
    question: str
    plan: str
    reflection: str
    tool_rounds: int
    max_steps: int
    use_reflection: bool

In [38]:
PLANNER_PROMPT = """คุณคือ planner ของ AI agent
สร้างแผน 2-5 ข้อสำหรับตอบคำถาม โดยระบุว่าต้องค้นหา ตรวจรายละเอียด หรือคำนวณอะไร
อย่าเดาคำตอบ อย่าแสดง chain-of-thought และยังไม่ต้องตอบคำถามผู้ใช้"""

def message_text(message) -> str:
    
    if isinstance(message.content, str):
        return message.content
        
    return "\n".join(
        block.get("text", "") for block in message.content
        if isinstance(block, dict) and block.get("text")
    )

def plan_node(state: AgentState):
    response = llm.invoke([
        SystemMessage(content=PLANNER_PROMPT),
        HumanMessage(content=state["question"]),
    ])
    return {"plan": message_text(response)}

In [39]:
REFLECTION_PROMPT = """คุณคือผู้ตรวจสอบความคืบหน้าของ AI agent
จากคำถาม แผน และผลจาก tools ให้สรุปไม่เกิน 3 บรรทัดว่า:
1) มีหลักฐานอะไรแล้ว 2) มี error หรือไม่ 3) ยังขาดอะไรหรือพร้อมสรุป
อย่าแก้โจทย์แทน agent และอย่าแสดง chain-of-thought"""

def reflect_node(state: AgentState):
    next_round = state["tool_rounds"] + 1
    if not state["use_reflection"]:
        return {"reflection": "ปิด reflection", "tool_rounds": next_round}

    observations = [
        {"tool": message.name, "result": message.content}
        for message in state["messages"] if isinstance(message, ToolMessage)
    ]
    
    context = to_json({
        "question": state["question"],
        "plan": state["plan"],
        "observations_so_far": observations,
    })
    
    response = llm.invoke([
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=context),
    ])
    return {"reflection": message_text(response), "tool_rounds": next_round}

## Step 4: ประกอบ ReAct Graph

`StateGraph` เชื่อม node ตามลำดับ `plan → agent → tools → reflect → agent` โดย `tools_condition` ตรวจ `AIMessage` ล่าสุด:

- ถ้ามี `tool_calls` ให้ route ไปยัง `ToolNode`
- ถ้าไม่มี tool call แสดงว่า Agent พร้อมตอบและ graph ไปยัง `END`

เมื่อใช้ tools ครบ `max_steps` แล้ว `agent_node` จะเรียก LLM โดยไม่ bind tools เพื่อบังคับให้สรุปจากหลักฐานที่มี


```mermaid
flowchart LR
    A[User Question] --> B[Plan]
    B --> C[Agent / Reason]
    C -->|tool_calls| D[Act: Run Tools]
    D --> E[Observe]
    E --> F[Reflect]
    F --> C
    C -->|no tool call| G[Final Answer]
    style C fill:#ff9999,stroke:#333
    style D fill:#ff9999,stroke:#333
    style G fill:#ff9999,stroke:#333
```

In [41]:
AGENT_PROMPT = """คุณคือ general-purpose agent ที่ตอบโดยอ้างอิงข้อมูลจาก tools
- ทำตามแผน แต่ปรับแผนได้เมื่อ Observation ไม่เป็นไปตามคาด
- เลือกใช้ tools เอง และเรียกหลาย tools พร้อมกันได้เมื่อไม่ขึ้นต่อกัน
- ห้ามเดา product_id ราคา rating stock หรือผลคำนวณ
- ถ้า tool มี error ให้แก้ input หรือลองวิธีอื่น
- ตอบจากหลักฐานที่ tools ส่งกลับมา และบอกข้อจำกัดเมื่อข้อมูลไม่พอ
- ตอบเป็นภาษาไทย

ไม่ต้องเปิดเผย chain-of-thought ให้แสดงเฉพาะคำตอบสุดท้ายที่กระชับ"""

def agent_node(state: AgentState):
    budget_exhausted = state["tool_rounds"] >= state["max_steps"]
    context = (
        f"{AGENT_PROMPT}\n\nแผนระดับสูง:\n{state['plan']}"
        f"\n\nReflection ล่าสุด:\n{state['reflection'] or 'ยังไม่มี'}"
    )
    
    if budget_exhausted:
        context += "\n\nใช้ tools ครบจำนวนรอบแล้ว ให้สรุปจากหลักฐานที่มีและบอกข้อมูลที่ยังขาด"
        model = llm
    else:
        model = llm_with_tools

    response = model.invoke([SystemMessage(content=context), *state["messages"]])
    return {"messages": [response]}

In [42]:
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(AgentState)
builder.add_node("plan", plan_node)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("reflect", reflect_node)

builder.add_edge(START, "plan")
builder.add_edge("plan", "agent")
builder.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "tools", "__end__": END},
)
builder.add_edge("tools", "reflect")
builder.add_edge("reflect", "agent")

agent_graph = builder.compile()

In [50]:
def run_agent(question: str, max_steps: int = 8, use_reflection: bool = True, verbose: bool = True):
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "question": question,
        "plan": "",
        "reflection": "",
        "tool_rounds": 0,
        "max_steps": max_steps,
        "use_reflection": use_reflection,
    }

    
    trace = {"question": question, "plan": "", "steps": []}

    # # Run it as sync request
    # final_state = agent_graph.invoke(
    #     initial_state,
    #     config={"recursion_limit": max_steps * 4 + 10},
    # )
    for update in agent_graph.stream(
        initial_state,
        config={"recursion_limit": max_steps * 4 + 10},
        stream_mode="updates",
    ):
        for node_name, output in update.items():
            trace, final_answer = display_output(node_name, output, trace, True)

    trace["final_answer"] = final_answer
    return {"answer": final_answer, "trace": trace}

In [51]:
def display_output(node_name, output, trace, verbose: bool = True):
    final_answer = "ไม่สามารถหาคำตอบได้"
    
    if node_name == "plan":
        trace["plan"] = output["plan"]
        if verbose:
            print("=== PLAN ===")
            print(output["plan"])

    elif node_name == "agent":
        message = output["messages"][-1]
        if message.tool_calls:
            if verbose:
                print("\n=== ACT ===")
                for call in message.tool_calls:
                    print(f"Action: {call['name']}({call['args']})")
        else:
            final_answer = message_text(message)
            if verbose:
                print("\n=== FINAL ANSWER ===")
                print(final_answer)

    elif node_name == "tools":
        observations = [
            {"tool": message.name, "result": message.content}
            for message in output["messages"]
        ]
        trace["steps"].append({"observations": observations})
        if verbose:
            print("\n=== OBSERVE (ToolNode) ===")
            for observation in observations:
                print(f"{observation['tool']}: {observation['result']}")

    elif node_name == "reflect":
        reflection = output["reflection"]
        if trace["steps"]:
            trace["steps"][-1]["reflection"] = reflection
        if verbose:
            print("\n=== REFLECT ===")
            print(reflection)
            
    return trace, final_answer

## Example 1: งานง่ายที่ใช้ Tool เดียว

Agent ควรวางแผนสั้น ๆ เรียก `calculator` และสรุปคำตอบจาก Observation

In [52]:
result = run_agent(
    "สินค้าราคา 250 บาท จำนวน 3 ชิ้น ต้องจ่ายทั้งหมดเท่าไร?",
    max_steps=4,
)

=== PLAN ===
แผนการทำงาน:
1. ระบุราคาต่อชิ้นของสินค้า (250 บาท) และจำนวนสินค้า (3 ชิ้น)
2. คำนวณราคารวมโดยนำราคาต่อชิ้นคูณด้วยจำนวนสินค้า ($250 \times 3$)

=== ACT ===
Action: calculator({'a': 250, 'operation': 'multiply', 'b': 3})

=== OBSERVE (ToolNode) ===
calculator: {"a": 250.0, "b": 3.0, "operation": "multiply", "result": 750.0}

=== REFLECT ===
1) มีหลักฐานผลคูณจากเครื่องคิดเลขว่า $250 \times 3 = 750$ บาท
2) ไม่มี Error เกิดขึ้น
3) ได้ข้อมูลครบถ้วนแล้ว พร้อมสรุปคำตอบสุดท้าย

=== FINAL ANSWER ===
ต้องจ่ายทั้งหมด 750 บาทค่ะ


In [53]:
result = run_agent(
    "100 - (43 + 56 - 32*4)*3 = ?",
    max_steps=10,
)

=== PLAN ===
1. คำนวณผลลัพธ์ในวงเล็บตามลำดับความสำคัญของทางคณิตศาสตร์ (คูณก่อน แล้วจึงนำมาบวกและลบ)
2. นำผลลัพธ์จากในวงเล็บไปคูณกับ 3
3. นำ 100 มาลบด้วยผลคูณที่ได้จากขั้นตอนที่ 2 เพื่อหาคำตอบสุดท้าย

=== ACT ===
Action: calculator({'b': 4, 'operation': 'multiply', 'a': 32})

=== OBSERVE (ToolNode) ===
calculator: {"a": 32.0, "b": 4.0, "operation": "multiply", "result": 128.0}

=== REFLECT ===
1) มีผลคูณของ 32 * 4 ได้ 128 แล้ว
2) ไม่มี error
3) ยังขาดการคำนวณในวงเล็บที่เหลือ (43 + 56 - 128), การคูณผลลัพธ์ด้วย 3 และการลบออกจาก 100

=== ACT ===
Action: calculator({'operation': 'add', 'a': 43, 'b': 56})

=== OBSERVE (ToolNode) ===
calculator: {"a": 43.0, "b": 56.0, "operation": "add", "result": 99.0}

=== REFLECT ===
1) มีผลคูณ (32*4 = 128) และผลบวก (43+56 = 99) แล้ว
2) ไม่มี error
3) ขาดการนำ 99 มาลบกับ 128 และดำเนินการต่อตามลำดับแผนเพื่อสรุปคำตอบสุดท้าย

=== ACT ===
Action: calculator({'b': 128, 'operation': 'subtract', 'a': 99})

=== OBSERVE (ToolNode) ===
calculator: {"a": 99.0, "b": 1

## Example 2: งานหลายขั้นตอนที่ใช้หลาย Tools

คำถามนี้บังคับให้ Agent ต้องค้นหา ID, อ่านรายละเอียดหลายสินค้า, เลือกราคาต่ำที่สุด, ใช้ calculator คำนวณราคาสองคู่ และ Reflect ว่ามีข้อมูลครบก่อนตอบ

In [45]:
result = run_agent(
    "ค้นหารองเท้า 3 รายการ เปรียบเทียบชื่อ ราคา และคะแนน แล้วแนะนำรายการที่ราคาต่ำที่สุด ถ้าซื้อ 2 คู่ต้องจ่ายเท่าไร โดยห้ามเดาข้อมูล",
    max_steps=8,
)

=== PLAN ===
แผนการทำงาน:

1. ค้นหารองเท้าจำนวน 3 รายการ พร้อมระบุชื่อ ราคา และคะแนนรีวิวจากแหล่งข้อมูลที่น่าเชื่อถือ
2. เปรียบเทียบข้อมูลชื่อ ราคา และคะแนนของรองเท้าทั้ง 3 รายการ
3. คัดเลือกและแนะนำรายการที่มีราคาต่ำที่สุด
4. คำนวณราคารวมสำหรับการซื้อรองเท้าที่มีราคาต่ำที่สุดจำนวน 2 คู่

=== ACT ===
Action: search_products({'keyword': 'รองเท้า'})

=== OBSERVE (ToolNode) ===
search_products: {"keyword": "รองเท้า", "count": 3, "products": [{"product_id": "20659907642", "title": "□❡รองเท้าแตะผู้หญิง Wnc Native Waterproof Shoes EVA Non-slip Beach Hollow Sandals Solid Color Couple"}, {"product_id": "19950623906", "title": "[ลูกค้าใหม่ราคา 1 บาท]🍎รองเท้านักเรียนโกลซิตี้ GCรุ่นFC001/Matin T205 ทนชาย หญิง น้ำตาล ขาว ดำ ไซร์31-45(มีบิลเบิกรรให้)"}, {"product_id": "25825858326", "title": "รองเท้าผ้าใบ KE-EN Original Running Shoes ZIONIC สีเขียว พร้อมส่ง"}]}

=== REFLECT ===
1) ได้รายการรองเท้า 3 รายการแล้ว แต่ยังไม่มีข้อมูลราคาและคะแนนของแต่ละรายการ
2) ยังไม่มี error ในข้อมูลปัจจุบัน
3) ขาดข้อม

### Inspect Agent Trace

`result["trace"]` เก็บ plan, observations, reflections และ final answer แยกจากข้อความสำหรับ API จึงนำไปทำ logging, debugging หรือ evaluation ต่อได้

In [47]:
print(json.dumps(result["trace"], ensure_ascii=False, indent=2))